# Capstone G1: Designing better CRISPR guides

Given a 20-letter guide, will the CRISPR scissors cut well there? You're building the tool that helps a scientist pick a good guide *before* running a slow, costly experiment.

**Your group's priority: interpretability.** A lab scientist won't trust a mystery box telling them which guide to order -- your model has to *show* it reads the DNA the way the biology actually works.

**Your goal (the rubric):** build it, check honestly how well it works, find a case where it fails, defend one choice you made, and make sure *both* partners can explain everything.

### How to use this notebook (read this first)

- A notebook is a stack of **cells**. A gray cell is **code**; this white cell is a note.
- To run a code cell, click it and press **Shift+Enter**. Run them **top to bottom, in order** --
  later cells use things earlier cells made.
- **Red text is normal**, not a disaster -- it's usually a warning. A real error stops with a
  message; read the last line, or paste it to Claude and ask "what does this mean?"
- You will not understand every line, and that's fine. But you should be able to say, in plain
  English, *what each cell is for*. Stuck on a line? Ask Claude: **"explain this cell line by line."**
- Nothing you click here can break anything. Experiment.

### You have Claude Code -- so here's your *real* job

You can already make a computer write code: just describe what you want to Claude Code and it will
build it. So **writing code is no longer the hard part.** The skill that actually makes you good at
this -- the thing this notebook is training -- is **judgment**:

1. **What are you trying to build?** "Accurate" isn't a goal. A cancer screen that must never miss a
   tumor is a *different tool* than one that's just right-on-average. Decide what "good" means first.
2. **What tools exist?** Different models, different ways to prep the data, different ways to grade a
   model. This notebook is a guided tour of that toolbox.
3. **Why does each tool exist** -- what problem it was invented to solve.
4. **How do you pick the right one?** That's the whole game. The wrong tool gives a wrong answer even
   with flawless code.

So: read the notes, play with the dials, and when you want to change or add something, **describe it
to Claude Code and let it write the code** -- then be ready to explain *why you chose it*. That "why"
is what you're graded on, and it's what a real scientist or doctor would ask you.

In [ ]:
# WHAT THIS DOES: gets the course files onto this Colab machine and installs the tools we need.
# (On your own computer it does nothing.) Just run it once and wait for "setup ok".
import os, sys
if not os.path.exists("../../capstone_common.py"):
    os.system("git clone -q https://github.com/jinchiwei/outset-ai-healthcare.git")
    os.chdir("outset-ai-healthcare/notebooks/day3_capstone/projects/g1_crispr")
sys.path.insert(0, "../..")            # so we can import the course helper files
sys.path.insert(0, "../../../_shared")
import colab_setup
colab_setup.ensure(*colab_setup.DAY3_SEQ)

### Plain-English vocabulary

- **CRISPR** is molecular scissors. You point it at a spot in DNA with a short **guide** (20 letters of
  A/C/G/T). Some guides cut well, some barely cut. You want to predict *which*, before spending weeks
  in the lab -- that's the tool you're building.
- A **model** is a rule the computer learns from examples of guides that did and didn't cut well.
- Here the data is **text** (DNA letters), not pixels or a spreadsheet -- so your first real decision
  is *how to turn letters into numbers*, because a computer can't do math on the letter "A".

In [ ]:
# WHAT THIS DOES: loads 5,310 real guides, each with a score for how well it cut.
import sys
sys.path.insert(0, "../..")
import capstone_seq as cs

df, meta = cs.load_guides()
print(meta["name"], "  ->", df.shape[0], "guides")
print(meta["citation"])
df[["guide30", "guide20", "gene", "gc_content", "efficiency", "high_efficiency"]].head()

### How to read one guide

Each row is a 30-letter stretch of DNA around one cut site:

```
[ 4 letters ][   20-letter guide   ][ NGG ][ 3 letters ]
  context        this is the guide     PAM     context
```

- The **guide** (middle 20) is the part you design.
- The **PAM** (the "NGG") is a landmark the scissors grab onto.
- **Key biology:** the scissors first grab the PAM, then check the ~10 letters *right next to it* --
  called the **seed**. If the seed doesn't match well, it won't cut. So letters near the PAM should
  matter most. **Remember this -- your model should rediscover it.**
- `efficiency` = how well it cut (0 to 1). `high_efficiency` = 1 for the best cutters, else 0.

### A first, no-math guess: GC content

**What this does:** checks whether one simple number -- the fraction of G's and C's in the guide --
lines up with efficiency. (G-C pairs stick together more tightly, so people suspected they'd matter.)
**Why bother:** it's a baseline. If a fancy model can't beat one obvious number, it isn't earning its keep.

In [ ]:
cs.gc_vs_efficiency(df)

### Turn letters into numbers -- two ways (this is the whole project)

A model needs numbers. There are two ways to convert a sequence, and the difference *is* the lesson:

- **one-hot** -- give every position 4 light-switches (A, C, G, T) and flip on the one that's there.
  This remembers **where** each letter sits. (Each position gets 4 switches, so the 20-letter guide
  becomes 20 x 4 = 80 numbers.)
- **k-mer** -- just **count** how many A/C/G/T (and letter-pairs) there are, ignoring order.

**The analogy:** "LISTEN" and "SILENT" have the *exact same letters*. k-mer counting can't tell them
apart -- it threw away the order. one-hot keeps the order. Since biology cares *where* a letter is
(the seed!), one-hot should win. Run it both ways below and see.

**How to choose a representation:** match the tool to the biology. If *order* matters -- and here it
does -- pick the one that keeps order (one-hot). If only the *amount* of each letter mattered, plain
counting (k-mer) would be simpler and just as good. Picking the representation *is* the science; the
model is almost an afterthought.

**One honest catch:** only about **1 in 5** guides is high-efficiency. So a lazy model that *always*
says "low" is right 80% of the time -- and completely useless. That's why we watch **AUC** instead of
accuracy: AUC ignores that cheat (0.5 = coin flip, 1.0 = perfect).

In [ ]:
from ipywidgets import interact_manual, Dropdown

def build(representation, sequence, model):
    cs.fit_eval_class(df, mode=representation, seq_col=sequence, model=model)

try:
    interact_manual(build,
        representation=Dropdown(options=["onehot", "kmer"], value="onehot", description="representation",
                                style={"description_width": "initial"}),
        sequence=Dropdown(options=["guide20", "guide30"], value="guide20", description="sequence"),
        model=Dropdown(options=list(cs.make_classifiers()), value="CatBoost", description="model"))
except ImportError:
    cs.fit_eval_class(df, mode="onehot", model="CatBoost")

**Do this:** run it once with **onehot**, once with **kmer**, same model. Watch the **AUC**. one-hot
should win -- and that's a real result: *the way you prepared the data mattered more than the model.*

### Predict the exact score instead of yes/no (regression)

"High vs low" throws away detail. The real target is a **number** from 0 to 1. Can the model predict it?

**How to read the score:** **R²** answers "how much of the ups-and-downs did the model actually
explain?" R² = 0 means "no better than always guessing the average"; 1.0 means perfect.

**Yes/no or a number -- how to choose?** Match it to the *decision you'll make*. If the choice is
binary (cut here or not), yes/no is enough. If you must **rank** many options -- say, pick the top 3
guides out of 100 to order -- you need the number. The task decides the tool, not the other way around.

In [ ]:
from ipywidgets import interact_manual, Dropdown

def build_reg(representation, model):
    cs.fit_eval_reg(df, mode=representation, model=model)

try:
    interact_manual(build_reg,
        representation=Dropdown(options=["onehot", "kmer"], value="onehot", description="representation",
                                style={"description_width": "initial"}),
        model=Dropdown(options=list(cs.make_regressors()), value="Random Forest", description="model"))
except ImportError:
    cs.fit_eval_reg(df, mode="onehot", model="Random Forest")

### Did the model learn real biology? (your headline slide)

Your priority is **interpretability** -- not just *is it right*, but *is it right for the right reason*.

**What this does:** trains a simple model and plots **how much each position** in the guide matters.
**Why it's the payoff:** if the tall bars land near the PAM end (the **seed** you read about above),
then your model figured out real CRISPR biology *on its own, from data* -- nobody told it about seeds.
A black box you can explain is a black box people will trust.

In [ ]:
cs.position_importance(df)

### Where to take it (ask Claude to help)
- Train on all genes but one, then test on the held-out gene -- does guide design carry over to a gene it never saw?
- Invent your own clue by hand (e.g. 'is there a G right before the PAM?') and see if it helps.
- Look up a real guide-design website (Benchling, CRISPOR) and compare its advice to your model's.